In [ ]:
import json
import re
from typing import List, Dict, Any
from gigachat_api import gc  # Ваше подключение к API

In [ ]:
METRIC_NAME = "Естественность"
SYSTEM_PROMPT = """
    Ты - судья по оценке последней реплики оператора банка на **естественность** для голосового общения.
    Верни строго json с ключами 'label' (1 - если финальная реплика оператора звучит человечно, естественно, без канцеляризмов, 
    0 - если последняя фраза оператора шаблонная.ю роботизированная или перегружена официальным языком) и 'reasoning' - краткий тезис, 
    почему такая оценка."""

FEW_SHOT_HISTORY = [
    {
        "role": "user",
        "content": "КОНТЕКСТ ДИАЛОГА:\n[Клиент]: Сколько это стоит?\n\nТЕКУЩИЙ ОТВЕТ ОПЕРАТОРА:\n[Оператор]: Стоимость составляет 10 тысяч рублей в месяц."
    },
    {
        "role": "assistant",
        "content": '{"обоснование": "Оператор ответил утверждением и не задал вопрос. Инициатива упущена.", "оценка": 0}'
    },
    {
        "role": "user",
        "content": "КОНТЕКСТ ДИАЛОГА:\n[Клиент]: Сколько это стоит?\n\nТЕКУЩИЙ ОТВЕТ ОПЕРАТОРА:\n[Оператор]: Стоимость от 10 тысяч. Подскажите, какой у вас объем данных?"
    },
    {
        "role": "assistant",
        "content": '{"обоснование": "Оператор перехватил инициативу встречным уточнением.", "оценка": 1}'
    }
]

In [ ]:
def preprocess_greenplum_text(raw_text: str):
    tokens = re.split(r'(CLIENT:|OPERATOR:)', raw_text)
    clean_lines = []
    current_tag = ""
    
    for token in tokens:
        token = token.strip()
        if not token:
            continue
        if token == "CLIENT:":
            current_tag = "[Клиент]"
        elif token == "OPERATOR:":
            current_tag = "[Оператор]"
        else:
             if current_tag:
                clean_lines.append(f"{current_tag}: {token}")
    return clean_lines

In [ ]:
def evaluate_one_dialogue(raw_text: str) -> float:

    lines = preprocess_greenplum_text(raw_text)
    dialogue_scores = []
    
    for i, line in enumerate(lines):
        if line.startswith("[Оператор]"):
            context_lines = lines[0:i]
            context_text = "\n".join(context_lines)
            
            query_for_eval = f"КОНТЕКСТ ДИАЛОГА (С НАЧАЛА):\n{context_text}\n\nТЕКУЩИЙ ОТВЕТ ОПЕРАТОРА ДЛЯ ОЦЕНКИ:\n{line}"
            
            try:
                messages, _ = gc(
                    query=query_for_eval,
                    system_prompt=SYSTEM_PROMPT,
                    chat_history=FEW_SHOT_HISTORY,
                    temperature=0.1
                )
                result = json.loads(messages[-1]["content"])
                dialogue_scores.append(int(result.get("оценка", -100)))

            except Exception:
                dialogue_scores.append(-20) 
                
    if not dialogue_scores:
        return -4
        
    return sum(dialogue_scores) / len(dialogue_scores)

In [ ]:
def run_eval_pipeline(file_path: str):
    all_dialogues_rates = []
    
    print(f"Чтение файла: {file_path}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
                
            dialogue_data = json.loads(line)
            score = evaluate_single_dialogue(dialogue_data["string_agg"])
            all_dialogues_rates.append(score)
            
    if not all_dialogues_rates:
        print("Файл пуст или данные не найдены.")
        return
    
    total_dataset_score = sum(all_dialogues_rates) / len(all_dialogues_rates)
    return total_dataset_score

In [ ]:
check = run_eval_pipeline("json_dialogues/test.jsonl")